<a href="https://colab.research.google.com/github/shrutioak06/Shruti_DACSS690C/blob/main/index.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<center><img src="https://github.com/DACSS-CSSmeths/guidelines/blob/main/pics/small_logo_ccs_meths.jpg?raw=true" width="700"></center>







-------


# Spatial AutoCorrelation.

<a target="_blank" href="https://colab.research.google.com/github/DACSS-CSSmeths-summer/spatial/blob/main/index.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>


## I. Getting ready


### Libraries needed

Let's verify:

In [ ]:
# !pip show pysal pandas geopandas libpysal esda

In [ ]:
## IF you need to install in Colab (this is temporal)
# !pip install pysal

### The Data

The URLS from GitHub

In [1]:
# Get used to save the data on GitHub
LinkUS="https://github.com/DACSS-CSSmeths-summer/datafiles/raw/main/cb_2023_us_state_500k.zip"


Reading the USA map:

In [ ]:
import geopandas as gpd

us_states = gpd.read_file(LinkUS)

In [ ]:
us_states

Checking **how** the map came:

* the crs (you may need to change this later)
* confirming crs is projected
* type of geometry in a row

In [ ]:
us_states.crs.to_epsg(),\
us_states.crs.is_projected, \
set(us_states.geom_type)

Taking a look:

In [ ]:
us_states.head()

In [ ]:
## we have a geodataframe
# Notice this map has very basic information per state
us_states.info()

Let's reproject the crs:

In [ ]:
us_states=us_states.to_crs('ESRI:102004')
us_states.plot()

**Column as row index:**

Let's use the state name **as index**, that would help an easier identification of the places when we see most outputs (otherwise we will see just numerical indexes) :

In [ ]:
# THIS IS NOT NEEDED, but useful
us_states.set_index('NAME', inplace=True)
us_states.head()

Let's keep some states, which will help understand the concepts visually in most examples:

In [ ]:
someStates=['Utah','Colorado','Arizona','New Mexico', 'Florida','Georgia','Alabama']
sub_us=us_states[us_states.index.isin(someStates)]
sub_us

## II. Neighborhood

### Who is my neighbor?


Activate PYSAL (*libpysal*), where we have functions to determine one polygon (`the focal`) find its neighbors.

In [ ]:
from libpysal.graph import Graph

## Set random seed for reproducibility:
from numpy.random import seed
seed(42)

### II.1 Contiguity

> If you touch my boundaries, you are my neighbor

#### II.1.A. Rook criterion

Let's use **sub_us** GDF to understand this criterion:

In [ ]:
rook_US=Graph.build_contiguity(sub_us,rook=True) # rook by default

Now let's check the ouput:

**a. adjacency**

In [ ]:
rook_US.adjacency # shows you the neighbors

The previous results shows only the neighbors of the focals, to recreate a wide format:

In [ ]:
import pandas as pd
pd.DataFrame(rook_US.adjacency).unstack()

We generally fill those missing values (not a neighbor) with zero.

In [ ]:
rook_US_Matrix=pd.DataFrame(rook_US.adjacency).unstack().fillna(0)
rook_US_Matrix

SInce we are going to use several times, we may create a custom procedure:

In [ ]:
toWideMatrix=lambda g:pd.DataFrame(g.adjacency).unstack().fillna(0)

Notice we created this wide matrix for pedagogical purposes. It is a bad idea if you have many columns and rows.

**b. adjacency graph**

As we have a `GRAPH`, we can identify these neighborhood relationships via edges:

In [ ]:
##this will be used several times:
general_arguments=dict(gdf=sub_us,node_kws=dict(color='red'), edge_kws=dict(alpha=0.4,color='blue'),zoom_start = 6)

rook_US.explore(**general_arguments)

#### II.1.B. Queen criterion


Let's see what we get:

**a. adjacency matrix**

In [ ]:
queen_US=Graph.build_contiguity(sub_us,rook=False)

# applying our procedure:

queen_US_Matrix=toWideMatrix(queen_US)
queen_US_Matrix

**b. adjacency plot**

In [ ]:
queen_US.explore(**general_arguments)


### II.2 Proximity

> If you are near enough, you could be my neigbor

Here, we need to know what is the point of reference to say you are close. It could be the `centroid`, or also the `representative point`:

In [ ]:
# distance between polygons just use these points.
# a good reason to have modified the CRS!

sub_us.representative_point()

In [ ]:
#then
us_sub_reference_points=sub_us.representative_point()


#### II.2.A. Nearest Neighbors


If assume K is 3:

**a. adjacency matrix**

In [ ]:
knn3_US = Graph.build_knn(us_sub_reference_points, # GDF
                                 k=3) # desired k

knn3_US_Matrix=toWideMatrix(knn3_US)
knn3_US_Matrix

**b. adjacency plot**

In [ ]:
knn3_US.explore(**general_arguments)

#### II.2.B Within zone of influence


Let's assume a 750 km distance band:

In [ ]:
band750k_US=Graph.build_distance_band(us_sub_reference_points, threshold=750000)

band750k_US_Matrix=toWideMatrix(band750k_US)
band750k_US_Matrix

In [ ]:
band750k_US.explore(**general_arguments)

## III. Normalization and Lags

### III. 1 Normalizing weights



Our matrices tell us which are neighbors, and some give us additional information related to the 'farness' of the identified neighbor. Let me compute the marginal values by row (sum):

In [ ]:
allMx=[rook_US_Matrix.sum(axis=1),
       queen_US_Matrix.sum(axis=1),
       knn3_US_Matrix.sum(axis=1),
       band750k_US_Matrix.sum(axis=1)]
pd.concat(allMx,axis=1)

We know that no-neighbors have value zero in the matrix. Then, non-zero (binary or non-binary) values carries some weight .
However, they are currently not normalized (they do not add to 1 by row), which may bias further procedures as raw values may 'distort' mathematical modelling.

For example, this binary matrix is non-normalized:

In [ ]:
queen_US_Matrix

But this  matrix is now normalized (by **r**ows):

In [ ]:
toWideMatrix(queen_US.transform("r"))

These new row-standardized matrices will serve a greater purpose: the computing of **spatial lags**.

### III. 2 Spatial Lag

**How are each of us doing related to...?**

This is "how each distrito is doing" related to some variable (social indicator or composite index) of the geometry. Since the USA map has no relevant no social column, let's use a map from Peru:

In [ ]:
LinkPeru="https://github.com/DACSS-CSSmeths-summer/datafiles/raw/main/PeruMaps.gpkg"
peru=gpd.read_file(LinkPeru,layer='good_geom')
# basic description
peru.info()

Besides the spatial units (DEPARTAMEN, PROVINCIA, DISTRITO, and Ubigeo - "Ubigeo" is a code ), you have:
 - **Poblacion**: Population (2017)
 - **IDH2019**: Human Development Index for DISTRITO (2019)                   
 - **Educ_sec_comp2019_pct**: Share of Population that finished High-School (2019)     
 - **NBI2017_pct**: Share of Population with poverty at the household level aggregated by DISTRITO. This index ("Unsatisfied Basic Needs") uses observable living conditions rather than income alone (2017).
 - **Viv_sin_serv_hig2017_pct**: Share of housing units that have no sanitation infrastructure aggregated by  DISTRITO (2017)



Take a look at the spatial properties:

In [ ]:
peru.crs.to_epsg(),\
peru.crs.is_projected, \
set(peru.geom_type)

Let's reproject and plot Peru's map:

In [ ]:
peru=peru.to_crs(5387)
peru.plot()




Notice we should not use the 'distrito' name as index for the whole country, because several of them are repeated:

In [ ]:
peru[peru['DISTRITO'].duplicated()]

Let's keep one DEPARTAMENTO: **Cusco**:

In [ ]:
cusco=peru[peru['DEPARTAMENTO']=='Cusco']
cusco

In [ ]:
# any duplicate names in distrito?
cusco[cusco['DISTRITO'].duplicated()]

In [ ]:
#then
cusco.set_index('DISTRITO', inplace=True)

We will use `Educ_sec_comp2019_pct`:

In [ ]:
cusco.Educ_sec_comp2019_pct.describe()

See the choropleth (quantile bining):

In [ ]:
cusco.plot(
    "Educ_sec_comp2019_pct",
    scheme="quantiles",
    cmap="Reds_r",
    legend=True,figsize=(12, 10))

In the map above, each distrito is showing its the value on the chosen variable. `NO NEIGHBOR` information has been used.

Compute the neighborhood matrix:

In [ ]:
cusco_queen=Graph.build_contiguity(cusco,rook=False)

When you gave no neighbors, you are an isolate (island). You can verify we do not have that:

In [ ]:
#  "islands" may create trouble.
# Should you keep using a _contiguity_ approach, you may consider filtering those out
cusco_queen.isolates



Now, let's get the normalized matrix:

In [ ]:
cusco_queen=cusco_queen.transform("r")

Then,  **the spatial lag of the variable** "Educ_sec_comp2019_pct" can be obtained like this:

In [ ]:
ylag = cusco_queen.lag(cusco["Educ_sec_comp2019_pct"])

Let me add it to the GDF:

In [ ]:
cusco=cusco.assign(Educ_sec_comp2019_pct_lagged=ylag)
cusco.head()

Let's plot HS share against its lag:

In [ ]:
cusco.plot.scatter("Educ_sec_comp2019_pct","Educ_sec_comp2019_pct_lagged")

Now we used the neighborhood information. Each _district_ (a dot) coordinate (x,y) tell you its education value(x) and the average education value of its neighbors (y), then:


*   Districts near  the top right corner are the ones with high education level whose neighbors also have, on average, high education level.
*   Districts near  the bottom left corner are the ones with low education level whose neighbors also have, on average, low education level.

Let go deeper in the next section.



## IV. Spatial autocorrelation

### IV. 1 Global

A correlation plot as the previous one should let us know if a variable is correlated with values of the neighbors (the lag), and if that were the case,  proximity is interfering statistical analysis, as the variable values are not independent.

The most well known measure to confirm that is the Moran's I index of global spatial autocorrelation. Let's see the value we get:


In [ ]:
import esda # from pysal

MoranGlobal_HS = esda.Moran(cusco['Educ_sec_comp2019_pct'], cusco_queen)

Once computed, you can retrieve its value and significance:

In [ ]:
MoranGlobal_HS.I,MoranGlobal_HS.p_sim

Moran’s I is interpreted like a Pearson r, but benchmarks are lower:

- |I| > 0.5 → **strong**; 0.2–0.5 → **moderate**; < 0.2 → **weak**.
- Sign indicates positive vs. negative spatial clustering.

Significance is assessed with a pseudo p-value from conditional randomisations of the data, because the theoretical null distribution is unknown for irregular spatial weights.

### IV. 2 Local

We can compute a Local Index of Spatial Association (LISA -local Moran) for each map polygon. That will help us find spatial clusters (spots) and spatial outliers:

- High-High (HH): values above average surrounded by values above average.These are also known as **hotSpot**s.
- Low-Low (LL): values below average surrounded by values below average. These are also known as **coldSpot**s.
- High-Low (HL): values above average surrounded by values below average.These are also known as **hotOutlier**s.
- Low-High (LH): values below average surrounded by values above average. These are also known as **coldOutlier**s.

It is also possible that no significant correlation is detected. Let's see those values:



a. Compute the LISAs:

In [ ]:
lisa = esda.Moran_Local(cusco['Educ_sec_comp2019_pct'], cusco_queen)


b. Plot the results (**fast way**)

In [ ]:
lisa.plot(cusco,crit_value=0.05,figsize=(10,13),legend=True)

#### IV. 2. 1 Analysis Local Spatial Correlation quadrants

You can not get the quadrant labels from the previous plot directly into the original GDF, thus complicating further analysis and exporting of those results.

First let's recover the labels into a new column:

In [ ]:
# get the quadrants
cusco['HS_lisa'] = lisa.get_cluster_labels(crit_value=0.05)

In [ ]:
# see the count
cusco['HS_lisa'].value_counts()

Now, we can prepare a comparisson:

In [ ]:
TheStats=["mean", "min", "max","var"]


# stats by the lisa cluster
grouped_stats = cusco.groupby('HS_lisa')['Educ_sec_comp2019_pct'].agg(TheStats)

# stats for the entire, ungrouped column
global_stats = cusco['Educ_sec_comp2019_pct'].agg(TheStats)

# Convert 'global_stats' (Series) to a DataFrame with an appropriate index name
global_stats_df = pd.DataFrame(global_stats).T
global_stats_df.index = ['Global/Total'] # simple but key

# Combine both
pd.concat([grouped_stats, global_stats_df])



From the results above, you may have several concerns and keep asking more questions:

- High-High clusters are concentrated areas of high percent of graduation from HighSchool. Is that expected in those areas? What has caused that? Has this been the same for a long time?

- Low-Low clusters are concentrated areas of low percent of graduation from HighSchool. What would it take to make a change in these areas? No doubt some kind of intervention is needed.

- High-Low clusters detect districts that outperform their neighbors. Can we leverage on those to improve the area? Are they in danger of worsening?

- Low-High clusters detect districts that are much worse than their neighbors, who are actually doing pretty well. Can we improve this place considering the surroundings need less intervention or not at all?

#### IV. 2. 2 Customizing Local Spatial Correlation quadrants

At this stage, we can rename the previous column to customize cluster labels and colors:

In [ ]:
##knowing
cusco['HS_lisa'].unique()

Keep in mind that the outliers may deserve more attention, so they may be at the top or bottom of the levels. You can add a number to achieve this purpose:

In [ ]:
oldLabels=['Insignificant', 'Low-Low', 'High-High', 'High-Low', 'Low-High']

# 3 will be at the middle
newLabels = ['3 no_pattern', '4 coldSpot','2 hotSpot','1 hotOutlier' , '5 coldOutlier']

labels = dict(zip(oldLabels, newLabels))

labels

Just using this dictionary to rename the column:

In [ ]:

cusco.replace({'HS_lisa':labels},inplace=True)

## see the count
cusco['HS_lisa'].value_counts().sort_index()

Finally, used that order to assign diverging color-blind safe palette:

In [ ]:
##custom colors- respect the previous order

import matplotlib.pyplot as plt
myColMap = plt.get_cmap('PuOr', 5)


##plot the map

cusco.plot(column='HS_lisa',
                categorical=True,
                cmap=myColMap,
                linewidth=0.1,
                edgecolor='k',
                legend=True,
                legend_kwds={'bbox_to_anchor': (0.3, 0.3)},
                figsize=(12,12))

In [ ]:
cusco.info()

If you prefer to plot in R's ggplot2, you may export this map, including the HS_lisa column:

In [ ]:
cusco.to_file("cusco.geojson")